Hands On - HR Policy Retrival RAG Systems

In [1]:
import os
from dotenv import load_dotenv

# LangChain Document Loaders & Splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings & Vector Store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# LLM (Groq)
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompts, Parsers & Runnables for LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

C:\Users\ANIK CHOWDHURY\Desktop\Tempr\ipykernel_28136\515125253.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
from pathlib import Path
from dotenv import find_dotenv


env_file = find_dotenv(".env", usecwd=True)
print("Loaded:", env_file)

load_dotenv(env_file, override=True)

assert os.getenv("GROQ_API_KEY") is not None, "GROQ_API_KEY Not Found"
assert os.getenv("GOOGLE_API_KEY") is not None, "GOOGLE_API_KEY Not Found"
assert os.getenv("HUGGINGFACEHUB_API_TOKEN") is not None, "HUGGINGFACEHUB_API_TOKEN Not Found"

Loaded: c:\Users\ANIK CHOWDHURY\Desktop\hr-policy-rag-system\.env


In [3]:
pdf_path = "../../ABC_HR_Policy_Manual.pdf" 

In [4]:
loader = PyPDFLoader(pdf_path)
docs = loader.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)
print(f"Total document chunks created: {len(splits)}")

Total document chunks created: 47


In [6]:
#embeddings = HuggingFaceEmbeddings(
#    model_name="sentence-transformers/all-MiniLM-L6-v2"
#)

#embeddings = HuggingFaceEndpointEmbeddings(
#    model="sentence-transformers/all-MiniLM-L6-v2"
#)


embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

In [7]:
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [9]:
llm = ChatGroq(
    temperature=0.5, 
    model_name="openai/gpt-oss-20b"
)


#llm = ChatGoogleGenerativeAI(
#    model = "gemini-3.6-flash"
#)

In [10]:
system_prompt = (
    "You are an assistant for HR policy question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Keep the answer concise, clear and professional."
    "\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

In [11]:
output_parser = StrOutputParser()

In [12]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [13]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [18]:
# পলিসি থেকে আপনার প্রশ্ন
question = "Can I take leave 8 days per month"

response = chain.invoke(question)

In [19]:
print("Question:", question)
print("\nAnswer:\n", response)

Question: Can I take leave 8 days per month

Answer:
 No.  
Under the current policy you can only take the entitlements listed for each leave type per calendar year (e.g., 8 days of Casual Leave, 10 days of Sick Leave, 18 days of Privilege Leave, etc.). Taking 8 days of any leave every month would exceed those limits. All leave must be applied for on PeopleHub and approved by your manager, and any unused leave cannot be carried forward or encashed.
